# PathSnap: 02. Candidate Matching Model Training
### Distributed Map-Matching & Route Reconstruction

This notebook trains and compares multiple machine learning algorithms across families:
1. **Candidate Road Arc Generation** via spatial KDTree indexing
2. **Feature Engineering** (orthogonal distance, heading difference, projection ratio, road segment length, GPS speed, distance rank)
3. **Model Training & Comparison** across 5 distinct algorithm families:
   - Linear/Logistic: Logistic Regression
   - Instance-based: K-Nearest Neighbors
   - Tree-based: Decision Tree
   - Ensemble: Random Forest
   - Boosting: Gradient Boosting
4. **Evaluation Metrics**:
   - Observation-level Top-1 Accuracy
   - Candidate Miss Rate
   - Macro F1-score & Log Loss
   - Route Continuity
5. **Bayesian Hyperparameter Optimization**
6. **Exporting Trained Model Artifact** (`artifacts/model.joblib`)


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path(".").resolve().parent))

from src.data.loader import TripLoader
from src.features.candidates import CandidateGenerator
from src.features.engineering import FeatureExtractor
from src.models.train import get_model_pipeline, save_model
from src.models.evaluate import evaluate_predictions, evaluate_observation_level
from src.models.tune import run_bayesian_tuning

loader = TripLoader("../data")
cg = CandidateGenerator(search_radius_m=120.0, top_k=6)
fe = FeatureExtractor()


## 1. Extract Training Features from Trajectories


In [ ]:
train_trip_ids = loader.list_trips("train")[:15]
dfs = []

for tid in train_trip_ids:
    trip = loader.load_trip(tid, split="train")
    track = trip["track"][:250]
    candidates = cg.generate_candidates(track, trip["arcs"], trip["nodes"])
    df = fe.extract_features_for_trip(track, trip["arcs"], trip["nodes"], candidates, route=trip["route"])
    df["trip_id"] = tid
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"Total candidate pair training records: {len(df_all)}")
print(f"Match class distribution: {df_all['is_match'].value_counts().to_dict()}")
df_all.head()


## 2. Load and Inspect Multi-Model Benchmark Results


In [ ]:
# Display consolidated model comparison results
comp_df = pd.read_csv("../results/model_comparison.csv")
comp_df


## 3. Visualize Model Comparison Metrics


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(comp_df["Model"], comp_df["Top-1 Accuracy"], color="#3B82F6")
ax.set_ylabel("Observation Top-1 Accuracy")
ax.set_title("Model Comparison: Observation Top-1 Accuracy")
ax.set_ylim(0, 1.0)
plt.xticks(rotation=20)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()


## 4. Bayesian Hyperparameter Optimization Results


In [ ]:
tuning_df = pd.read_csv("../results/tuning_results.csv")
print("Top 5 Optimization Trials:")
tuning_df.sort_values(by="mean_macro_f1", ascending=False).head()


## 5. Artifact Verification


In [ ]:
model_path = Path("../artifacts/model.joblib")
print(f"Model artifact exists: {model_path.exists()} (size: {model_path.stat().st_size} bytes)")
